In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from LoadData import *
from SystHelper import *
from CommonTrain import *
from Model import Seq

2023-10-26 15:21:43.309822: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Welcome to JupyROOT 6.28/06


# Set config

In [2]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]

# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1]
ERA = sys.argv[2]
REGION = sys.argv[3]
PART = sys.argv[4]


# # for dev
# VERSION = '20231021mujets-epoch250'
# # CHANNEL="ee"
# # INPUTDIR = setting["INPUTDIR"]+'/em18/'
# ERA = "em17"
# INPUTDIR = '/eos/user/h/hkwon/Run2/Preselection/20231021mujets/DUMP'
# # INPUTDIR = '/eos/user/h/hkwon/SWAN_projects/Zpeak/20230722jer_18/Presel/DUMP'
# REGION = 'SR'
# PART = 'benchmark'

20230430wp80_ANv3


# Load data

In [3]:
dic_df_raw, dic_nested_shape_sys = get_dic_raw_sys(ERA, INPUTDIR, REGION)

nested_dic_unsort = dic_nested_shape_sys
nested_dic_unsort['raw'] = dic_df_raw

# # temp!!
for key in nested_dic_unsort['met_u']:
    df = nested_dic_unsort['met_u'][key]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        nested_dic_unsort['met_u'][key] = df.drop(i)
    df = nested_dic_unsort['met_d'][key]
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        nested_dic_unsort['met_d'][key] = df.drop(i)
        
selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])

nested_dic_selected_columns = {}
nested_dic_selected_columns['raw'] = {}

shape_sys_list = ['met', 'met_jer', 'met_ue', 'jes', 'jer', 'scale']

for i, sys in enumerate(shape_sys_list):
    nested_dic_selected_columns[sys+'_u'] = {}
    nested_dic_selected_columns[sys+'_d'] = {}
    for key in nested_dic_unsort['raw']:
        nested_dic_selected_columns[sys+'_u'][key] = nested_dic_unsort[sys+'_u'][key][selected_columns]
        nested_dic_selected_columns[sys+'_d'][key] = nested_dic_unsort[sys+'_d'][key][selected_columns]
        if i==0:
            nested_dic_selected_columns['raw'][key] = nested_dic_unsort['raw'][key][selected_columns]

In [4]:
pd.set_option('display.max_columns', None)
nested_dic_unsort['raw']["Zp-2500_CH-345"]

,PuppiMET_pre,mass_pre,pt_lead_pre,pt_trail_pre,U_pre,JZB_pre,dPhill_pre,dPhilMET_pre,MT_pre,MT2_pre,Zpt_pre,dRll_pre,dPhillMET_pre,eta_lead_pre,eta_trail_pre,njet_pre,nbjetflav_pre,nbjetflav_jesup_pre,nbjetflav_jesdown_pre,nbjetflav_jerup_pre,nbjetflav_jerdown_pre,HT_pre,PuppiMET_phi_pre,MET_pre,MET_phi_pre,phi_lead_pre,phi_trail_pre,hemveto,genweight_u_pre,genweight_d_pre,genweight_pre,puweight_pre,puweight_u_pre,puweight_d_pre,l1prefiring_pre,l1prefiring_u_pre,l1prefiring_d_pre,effrecoSF_pre,effrecoSF_u_pre,effrecoSF_d_pre,effSF_pre,effSF_u_pre,effSF_d_pre,trigSF_pre,trigSF_u_pre,trigSF_d_pre,topSF_pre,topSF_u_pre,topSF_d_pre,btagSF_pre,btagSFbc_u_pre,btagSFbc_d_pre,btagSFbc_correlated_u_pre,btagSFbc_correlated_d_pre,btagSFlight_u_pre,btagSFlight_d_pre,btagSFlight_correlated_u_pre,btagSFlight_correlated_d_pre,btagSF_jes_u_pre,btagSF_jes_d_pre,btagSF_jer_u_pre,btagSF_jer_d_pre,pdf0_pre,pdf1_pre,pdf2_pre,pdf3_pre,pdf4_pre,pdf5_pre,pdf6_pre,pdf7_pre,pdf8_pre,pdf9_pre,pdf10_pre,pdf11_pre,pdf12_pre,pdf13_pre,pdf14_pre,pdf15_pre,pdf16_pre,pdf17_pre,pdf18_pre,pdf19_pre,pdf20_pre,pdf21_pre,pdf22_pre,pdf23_pre,pdf24_pre,pdf25_pre,pdf26_pre,pdf27_pre,pdf28_pre,pdf29_pre,pdf30_pre,pdf31_pre,pdf32_pre,pdf33_pre,pdf34_pre,pdf35_pre,pdf36_pre,pdf37_pre,pdf38_pre,pdf39_pre,pdf40_pre,pdf41_pre,pdf42_pre,pdf43_pre,pdf44_pre,pdf45_pre,pdf46_pre,pdf47_pre,pdf48_pre,pdf49_pre,pdf50_pre,pdf51_pre,pdf52_pre,pdf53_pre,pdf54_pre,pdf55_pre,pdf56_pre,pdf57_pre,pdf58_pre,pdf59_pre,pdf60_pre,pdf61_pre,pdf62_pre,pdf63_pre,pdf64_pre,pdf65_pre,pdf66_pre,pdf67_pre,pdf68_pre,pdf69_pre,pdf70_pre,pdf71_pre,pdf72_pre,pdf73_pre,pdf74_pre,pdf75_pre,pdf76_pre,pdf77_pre,pdf78_pre,pdf79_pre,pdf80_pre,pdf81_pre,pdf82_pre,pdf83_pre,pdf84_pre,pdf85_pre,pdf86_pre,pdf87_pre,pdf88_pre,pdf89_pre,pdf90_pre,pdf91_pre,pdf92_pre,pdf93_pre,pdf94_pre,pdf95_pre,pdf96_pre,pdf97_pre,pdf98_pre,pdf99_pre,pdf100_pre,psweight1_pre,psweight2_pre,psweight3_pre,psweight4_pre,scale1_pre,scale2_pre,scale3_pre,scale4_pre,scale5_pre,scale6_pre,pt_mu_pre,pt_el_pre,eta_mu_pre,eta_el_pre,phi_mu_pre,phi_el_pre,weight,hemfilter,wopu,wol1,woeff,wotop,wobsf,weighthdampUp,weighthdampDown,weightPSUp,weightPSDown,weightJECUp,weightJECDown,weightJERUp,weightJERDown,weightpuweightUp,weightpuweightDown,weightl1prefiringUp,weightl1prefiringDown,weighteffSFUp,weighteffSFDown,weighteffrecoSFUp,weighteffrecoSFDown,weighttopSFUp,weighttopSFDown,weightgenweightUp,weightgenweightDown,weighttrigSFUp,weighttrigSFDown,weightbtagSFbcUp,weightbtagSFbcDown,weightbtagSFbc_correlatedUp,weightbtagSFbc_correlatedDown,weightbtagSFlightUp,weightbtagSFlightDown,weightbtagSFlight_correlatedUp,weightbtagSFlight_correlatedDown
0,123.797653,379.616730,143.772629,130.493484,32.392056,-87.255074,2.243607,2.384674,397.464630,117.393936,119.647133,2.899485,2.876861,1.225830,-0.610809,0,0,0,0,0,0,0.000000,2.330917,122.185913,2.135742,-0.053757,-2.297363,1.0,1.0,1.0,1.0,0.976553,0.901406,1.044626,0.999023,0.999275,0.998772,0.994934,0.999898,0.989971,0.984238,0.992871,0.975639,0.998323,0.999137,0.997475,1.0,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.835815,0.832733,0.834198,0.842682,0.834442,0.843170,0.836090,0.835358,0.832001,0.830536,0.843170,0.847137,0.828888,0.842346,0.829102,0.844574,0.840149,0.837280,0.833282,0.840973,0.849670,0.829834,0.847778,0.834290,0.844696,0.826202,0.830841,0.828461,0.839478,0.846588,0.836304,0.835510,0.849609,0.830109,0.837097,0.829193,0.832855,0.844147,0.842865,0.841187,0.831482,0.832397,0.833862,0.825012,0.827698,0.832825,0.840851,0.833527,0.825043,0.839111,0.837189,0.840118,0.841492,0.832825,0.835754,0.839050,0.839600,0.830566,0.828278,0.833344,0.839325,0.839111,0.833221,0.821594,0.832977,0.828217,0.838104,0.833832,0.832214,0.836151,0.832977,0.830353,0.840576,0.843048,0.828979,0.838562,0.835815,0.831238,0.845184,0.836029,0.841522,0.835205,0.826019,0.833893,0.837860,0.827393,0.831635,0.837616,0.831787,0.833801,0.838837,

In [5]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [6]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [7]:
from multiprocessing import Pool
out_dir = "Outputs/DNN/"+ERA+"/"+REGION

def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            
#     year = ERA.replace('mm', '')
#     year = year.replace('ee', '')
#     year = year.replace('em', '')

    #-- implement systematics --#   
    os.system("mkdir -p "+out_dir)
    os.system("mkdir -p "+out_dir+"/AN")
    os.system("mkdir -p "+out_dir+"/sys")
    os.system("mkdir -p "+out_dir+"/root_files")
#     sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "trigSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.6
    else:
        threshold = 1    
    
    # draw plot with syst
#     dic_sys_sqr = draw_with_systematics(model_mass, dic_predict_sort, dic_nested_shape_sys, sys_list, "score"+model_mass, r, "region_dummy")
    file_prefix = out_dir
    sys_list = get_sys_list()
    dic_sys_sqr = get_dic_sys(model_mass, ERA, nested_dic_predict_sort['raw'], dic_nested_shape_sys, sys_list, "score"+model_mass, r, REGION, threshold, file_prefix, file_str=VERSION)
#     dic_sys_sqr = None # temp!

    hist=HistStack(nested_dic_predict_sort['raw'], "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name=out_dir+"/AN/"+model_mass+VERSION, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name=out_dir+"/pie_"+model_mass+VERSION)     
    hist.draw_pie_lastbin(file_name=out_dir+"/pie_"+model_mass+VERSION)     
    hist.export_yields_table(file_name=out_dir+"/"+model_mass+VERSION)     
     
    return None

# Load model and evaluate

In [8]:
import time

nbin = 20
if REGION == "VR":
    nbin = 20
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

if 'mm' in ERA:
    CHANNEL = 'mm'
if 'ee' in ERA:
    CHANNEL = 'ee'
if 'em' in ERA:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
# "ALL_MASSES": ["Zp-1700_CH-345", "Zp-1700_CH-595", "Zp-1700_CH-845", "Zp-2100_CH-345", "Zp-2100_CH-595", "Zp-2100_CH-845", "Zp-2500_CH-345", "Zp-2500_CH-595", "Zp-2500_CH-845", "Zp-2900_CH-345", "Zp-2900_CH-595", "Zp-2900_CH-845", "Zp-2900_CH-1095", "Zp-2900_CH-1345", "Zp-3300_CH-345", "Zp-3300_CH-595", "Zp-3300_CH-845", "Zp-3300_CH-1095", "Zp-3300_CH-1345", "Zp-3300_CH-1595",  "Zp-3700_CH-345", "Zp-3700_CH-595", "Zp-3700_CH-845", "Zp-3700_CH-1095", "Zp-3700_CH-1345", "Zp-3700_CH-1595", "Zp-3700_CH-1845", "Zp-4100_CH-345", "Zp-4100_CH-595", "Zp-4100_CH-845", "Zp-4100_CH-1095", "Zp-4100_CH-1345", "Zp-4100_CH-1595", "Zp-4100_CH-1845"]

model_masses = [m for m in dic_df_raw if 'Zp' in m] # if want to run all
if PART=="one":
    model_masses = ["Zp-1700_CH-345", "Zp-1700_CH-595", "Zp-1700_CH-845", "Zp-2100_CH-345", "Zp-2100_CH-595", "Zp-2100_CH-845", "Zp-2500_CH-345", "Zp-2500_CH-595"]
if PART=="two":
    model_masses = ["Zp-2500_CH-845", "Zp-2900_CH-345", "Zp-2900_CH-595", "Zp-2900_CH-845", "Zp-2900_CH-1095", "Zp-2900_CH-1345", "Zp-3300_CH-345", "Zp-3300_CH-595"]
if PART=="three":
    model_masses = ["Zp-3300_CH-845", "Zp-3300_CH-1095", "Zp-3300_CH-1345", "Zp-3300_CH-1595",  "Zp-3700_CH-345", "Zp-3700_CH-595", "Zp-3700_CH-845", "Zp-3700_CH-1095", "Zp-4100_CH-1845"]
if PART=="four":
    model_masses = ["Zp-3700_CH-1345", "Zp-3700_CH-1595", "Zp-3700_CH-1845", "Zp-4100_CH-345", "Zp-4100_CH-595", "Zp-4100_CH-845", "Zp-4100_CH-1095", "Zp-4100_CH-1345", "Zp-4100_CH-1595"]
if PART=="benchmark":
    model_masses = ['Zp-2500_CH-345']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict_modelmass = {}

key_list = list(nested_dic_selected_columns['raw'].keys())
key_list.remove('HWW')
key_list.remove('H')

# masses loop
for model_mass in model_masses:
    
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]

#     file_postfix = 'test' 
    
    nested_dic_tmp = {}
    nested_dic_scaled = {}
    nested_dic_tmp['raw'] = {}      
    nested_dic_scaled['raw'] = {}      
    for i, sys in enumerate(shape_sys_list):
        nested_dic_tmp[sys+'_u'] = {}
        nested_dic_tmp[sys+'_d'] = {}        
        nested_dic_scaled[sys+'_u'] = {}
        nested_dic_scaled[sys+'_d'] = {}

#         for key in nested_dic_selected_columns['raw']:
        for key in key_list:
            if i==0:
                nested_dic_tmp['raw'][key] = nested_dic_selected_columns['raw'][key].copy(deep=True)
            nested_dic_tmp['raw'][key]['zp_m'] = np.full(nested_dic_selected_columns['raw'][key].shape[0], int(zp_m))
            nested_dic_tmp['raw'][key]['ch_m'] = np.full(nested_dic_selected_columns['raw'][key].shape[0], int(ch_m))
            nested_dic_tmp[sys+'_u'][key] = nested_dic_selected_columns[sys+'_u'][key].copy(deep=True)
            nested_dic_tmp[sys+'_d'][key] = nested_dic_selected_columns[sys+'_d'][key].copy(deep=True)
            
            nested_dic_tmp[sys+'_u'][key]['zp_m'] = np.full(nested_dic_selected_columns[sys+'_u'][key].shape[0], int(zp_m))
            nested_dic_tmp[sys+'_d'][key]['zp_m'] = np.full(nested_dic_selected_columns[sys+'_d'][key].shape[0], int(zp_m))
            nested_dic_tmp[sys+'_u'][key]['ch_m'] = np.full(nested_dic_selected_columns[sys+'_u'][key].shape[0], int(ch_m))
            nested_dic_tmp[sys+'_d'][key]['ch_m'] = np.full(nested_dic_selected_columns[sys+'_d'][key].shape[0], int(ch_m))            

            if sys == 'raw':
                if nested_dic_tmp[sys][key].empty:
                    continue
#             if sys == 'met_ue' or sys == 'met':
            else:
                if nested_dic_tmp[sys+'_u'][key].empty:
                    continue
            pd.set_option('display.max_columns', None)
            
            nested_dic_scaled[sys+'_u'][key] = Transformer.transform(nested_dic_tmp[sys+'_u'][key]) 
            nested_dic_scaled[sys+'_d'][key] = Transformer.transform(nested_dic_tmp[sys+'_d'][key]) 
            if i==0:
                nested_dic_scaled['raw'][key] = Transformer.transform(nested_dic_tmp['raw'][key])            

    # predict
    nested_dic_predict = {}
    nested_dic_predict['raw'] = {}

    for i, sys in enumerate(shape_sys_list):
        nested_dic_predict[sys+'_u'] = {}
        nested_dic_predict[sys+'_d'] = {}
        for key in nested_dic_scaled['raw']:
            if i==0:
                nested_dic_predict['raw'][key] = model.predict(nested_dic_scaled['raw'][key], batch_size=nested_dic_scaled['raw'][key].shape[0])  
            if not key in nested_dic_scaled[sys+'_u']:
                continue    
            print(sys)    
            nested_dic_predict[sys+'_u'][key] = model.predict(nested_dic_scaled[sys+'_u'][key], batch_size=nested_dic_scaled[sys+'_u'][key].shape[0])
            nested_dic_predict[sys+'_d'][key] = model.predict(nested_dic_scaled[sys+'_d'][key], batch_size=nested_dic_scaled[sys+'_d'][key].shape[0])

    
    nested_dic_predict_modelmass[model_mass] = nested_dic_predict

# array in y-axis to dataframe
nested_dic_predict_sort = {}
nested_dic_predict_sort['raw'] = {}

for i, sys in enumerate(shape_sys_list):
    nested_dic_predict_sort[sys+'_u'] = {}
    nested_dic_predict_sort[sys+'_d'] = {}
#     for key in nested_dic_scaled['raw']:
    for key in key_list:
        if i==0:
            new_df = nested_dic_unsort['raw'][key].copy(deep=True).reset_index(drop="True")
        new_df_u = pd.DataFrame()
        new_df_d = pd.DataFrame()
        if sys == 'jes': # dirty cosmetic...
            new_df_u['weightJECUp'] = nested_dic_unsort[sys+'_u'][key]['weightJECUp'].copy(deep=True).reset_index(drop="True")
            new_df_d['weightJECDown'] = nested_dic_unsort[sys+'_d'][key]['weightJECDown'].copy(deep=True).reset_index(drop="True")
        elif sys == 'jer': # dirty cosmetic...
            new_df_u['weightJERUp'] = nested_dic_unsort[sys+'_u'][key]['weightJERUp'].copy(deep=True).reset_index(drop="True")
            new_df_d['weightJERDown'] = nested_dic_unsort[sys+'_d'][key]['weightJERDown'].copy(deep=True).reset_index(drop="True")
        elif sys == 'met_jer':
            new_df_u['weightmetjerUp'] = nested_dic_unsort[sys+'_u'][key]['weightmetjerUp'].copy(deep=True).reset_index(drop="True")
            new_df_d['weightmetjerDown'] = nested_dic_unsort[sys+'_d'][key]['weightmetjerDown'].copy(deep=True).reset_index(drop="True")
        elif sys == 'met_ue':
            new_df_u['weightmetueUp'] = nested_dic_unsort[sys+'_u'][key]['weightmetueUp'].copy(deep=True).reset_index(drop="True")
            new_df_d['weightmetueDown'] = nested_dic_unsort[sys+'_d'][key]['weightmetueDown'].copy(deep=True).reset_index(drop="True")
        else: # dirty cosmetic...
            new_df_u['weight'+sys+'Up'] = nested_dic_unsort[sys+'_u'][key]['weight'+sys+'Up'].copy(deep=True).reset_index(drop="True")
            new_df_d['weight'+sys+'Down'] = nested_dic_unsort[sys+'_d'][key]['weight'+sys+'Down'].copy(deep=True).reset_index(drop="True")
            
        nested_dic_predict_sort[sys+'_u'][key] = new_df_u
        nested_dic_predict_sort[sys+'_d'][key] = new_df_d
        
        if i==0:
            nested_dic_predict_sort['raw'][key] = new_df
    
for model_mass in model_masses:
    for i, sys in enumerate(shape_sys_list):
#         for key in nested_dic_scaled['raw']:
        for key in key_list:
            if i==0:
                nested_dic_predict_sort['raw'][key]["score"+model_mass] = nested_dic_predict_modelmass[model_mass]['raw'][key][:,0]
            nested_dic_predict_sort[sys+'_u'][key]["score"+model_mass] = nested_dic_predict_modelmass[model_mass][sys+'_u'][key][:,0]
            nested_dic_predict_sort[sys+'_d'][key]["score"+model_mass] = nested_dic_predict_modelmass[model_mass][sys+'_d'][key][:,0]
        
dic_nested_shape_sys={}
for i, sys in enumerate(shape_sys_list):
    dic_nested_shape_sys[sys+'_u']=nested_dic_predict_sort[sys+'_u']
    dic_nested_shape_sys[sys+'_d']=nested_dic_predict_sort[sys+'_d']
         
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1698326550.3579855


/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: Use

/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: Use

/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: Use

/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/cvmfs/sft.cern.ch/lcg/views/LCG_104a_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/sklearn/base.py:432: Use

...

1/1 [==============================] - 0s 54ms/step
jes
1/1 [==============================] - 0s 46ms/step
jes
1/1 [==============================] - 0s 38ms/step
jes
1/1 [==============================] - 0s 45ms/step
jes
1/1 [==============================] - 0s 47ms/step
jes
1/1 [==============================] - 0s 43ms/step
jes
1/1 [==============================] - 0s 54ms/step
jes
1/1 [==============================] - 0s 42ms/step
jes
1/1 [==============================] - 0s 43ms/step
jes
1/1 [==============================] - 0s 44ms/step
jes
1/1 [==============================] - 0s 40ms/step
jes
1/1 [==============================] - 0s 45ms/step
jes
1/1 [==============================] - 0s 51ms/step
jes
1/1 [==============================] - 0s 41ms/step
jes
1/1 [==============================] - 0s 42ms/step
jes
1/1 [==============================] - 0s 42ms/step
jes
1/1 [==============================] - 0s 42ms/step
jes
1/1 [==============================] - 0s 58ms/s

        0.26390817, -0.05410219,  0.49714597,  0.43335847, -0.05772048])}
{'scaleUp': array([4.93372938e+01, 1.44684988e+02, 2.02683588e+02, 0.00000000e+00,
       8.18376072e+00, 1.71025272e+01, 8.42890747e-01, 6.25746026e-02,
       7.09454871e-01, 1.24077498e+00, 1.03165260e+01, 4.32431758e+00,
       0.00000000e+00, 1.74480973e+00, 2.87639702e-01, 6.15615955e+00,
       6.47694717e-03, 2.69855988e+00, 5.02300988e+00, 0.00000000e+00]), 'scaleDown': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 2.04579953e+01,
       0.00000000e+00, 0.00000000e+00, 3.01759590e+00, 1.74794645e-01,
       1.20110068e-01, 0.00000000e+00, 2.59525206e-01, 0.00000000e+00,
       2.90314839e+00, 0.00000000e+00, 5.60062873e-02, 0.00000000e+00,
       2.92704747e-03, 0.00000000e+00, 0.00000000e+00, 2.65585216e-02]), 'puweightUp': array([4.95970923e+00, 9.55286536e+01, 7.44024221e+00, 1.03156397e+01,
       1.11366208e-01, 1.16268153e-01, 0.00000000e+00, 5.48885410e-01,
       2.97772999e+00, 3.832897

       4.25204770e-03, 8.23045115e-03, 5.99787687e-03, 1.95653572e-03])}
{'scaleUp': array([4.93372938e+01, 1.44684988e+02, 2.02683588e+02, 0.00000000e+00,
       8.18376072e+00, 1.71025272e+01, 8.42890747e-01, 6.25746026e-02,
       7.09454871e-01, 1.24077498e+00, 1.03165260e+01, 4.32431758e+00,
       0.00000000e+00, 1.74480973e+00, 2.87639702e-01, 6.15615955e+00,
       6.47694717e-03, 2.69855988e+00, 5.02300988e+00, 0.00000000e+00]), 'scaleDown': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 2.04579953e+01,
       0.00000000e+00, 0.00000000e+00, 3.01759590e+00, 1.74794645e-01,
       1.20110068e-01, 0.00000000e+00, 2.59525206e-01, 0.00000000e+00,
       2.90314839e+00, 0.00000000e+00, 5.60062873e-02, 0.00000000e+00,
       2.92704747e-03, 0.00000000e+00, 0.00000000e+00, 2.65585216e-02]), 'puweightUp': array([4.95970923e+00, 9.55286536e+01, 7.44024221e+00, 1.03156397e+01,
       1.11366208e-01, 1.16268153e-01, 0.00000000e+00, 5.48885410e-01,
       2.97772999e+00, 3.8328976

         6.71309852,   6.39827002,   5.22857552,   3.94959398])}
scaleUp
scaleDown
puweightUp
puweightDown
l1prefiringUp
l1prefiringDown
effrecoSFUp
effrecoSFDown
effSFUp
effSFDown
topSFUp
topSFDown
metUp
metDown
metjerUp
metjerDown
metueUp
metueDown
trigSFUp
trigSFDown
hdampUp
hdampDown
PSUp
PSDown
jesUp
jesDown
jerUp
jerDown
btagSFbcUp
btagSFbcDown
btagSFbc_correlatedUp
btagSFbc_correlatedDown
btagSFlightUp
btagSFlightDown
btagSFlight_correlatedUp
btagSFlight_correlatedDown
sum_up
sum_down
['scale', 'puweight', 'l1prefiring', 'effrecoSF', 'effSF', 'topSF', 'met', 'metjer', 'metue', 'trigSF', 'hdamp', 'PS', 'jes', 'jer', 'btagSFbc', 'btagSFbc_correlated', 'btagSFlight', 'btagSFlight_correlated']
array('f', [18.997133255004883, 0.07527294754981995, 82.40025329589844, 998.6488037109375, 688.6935424804688, 4449.552734375, 70.09491729736328, 36.12776565551758])
test
array('f', [1.4093629121780396, 0.010718097910284996, 1.4240325689315796, 31.012432098388672, 5.0370192527771, 7.71010875701

Info in <TCanvas::Print>: pdf file Outputs/DNN/em17/SR/norm_20231021mujets-epoch250_scoreZp-2500_CH-345breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/DNN/em17/SR/norm_20231021mujets-epoch250_scoreZp-2500_CH-345.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/DNN/em17/SR/norm_20231021mujets-epoch250_sign_scoreZp-2500_CH-345.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outputs/DNN/em17/SR/sys/systUp_puweight20231021mujets-epoch250_scoreZp-2500_CH-345breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/DNN/em17/SR/sys/systUp_puweight20231021mujets-epoch250_scoreZp-2500_CH-345.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/DNN/em17/SR/sys/systUp_puweight20231021mujets-epoch250_sign_scoreZp-2500_CH-345.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outpu

# Theory unc.

In [9]:
def write_sys_hist(h1, h2):
    file_name = out_dir+"/root_files/shape_v1_"+VERSION+model_mass
    f = rt.TFile(file_name+"_v1.root","UPDATE") 
    f.cd("signal_region")
#     h1.Write("zp_"+sys+"Up", rt.TObject.kWriteDelete)
#     h2.Write("zp_"+sys+"Down", rt.TObject.kWriteDelete)
    h1.Write()
    h2.Write()  

In [ ]:
from collections import defaultdict
import math

c = rt.TCanvas()
c.SetGrid()
rt.gStyle.SetOptStat(0)
    
processes = ['TT', 'WW', 'zp']    
# def export_pdf_scale_unc(model_mass): 
for p in processes:
    for model_mass in model_masses:
        if p=='zp':
            key = model_mass
            nset = 101
            score_key = 'score'+model_mass
            str_h = 'zp'
        else:
            key = p
            nset = 101
            score_key = 'score'+model_mass
            if p == 'TT':
                str_h = 'ttbar'
            if p == 'WW':
                str_h = 'ww'
                
        ##-- pdf unc. --##        
        w0 = nested_dic_predict_sort['raw'][key]['weight'].copy(deep=True)
        hists = []       
        for j in range(nset):
            h = rt.TH1D("", "", nbin, 0, 1)
            for i, v in enumerate(nested_dic_predict_sort['raw'][key][score_key]):
                w = w0[i]*nested_dic_predict_sort['raw'][key]['pdf'+str(j)+'_pre'][i]
                h.Fill(v, w)
        #     h.Draw("same")
            hists.append(h)
        # print(hists)
    #     c.Draw()
    #     c.SaveAs("Syst/pdfs_"+model_mass+".pdf")

        dic_yields=defaultdict(list)
        for i, h in enumerate(hists):
            for nb in range(nbin):
                dic_yields["pdf"+str(i)].append(h.GetBinContent(nb+1))

        unc_pdfs = []
        for nb in range(nbin):
            detla_sqrs = []
            for i in range(nset-1):
                delta = dic_yields["pdf"+str(i+1)][nb] - dic_yields["pdf0"][nb]
                detla_sqr = delta*delta
                detla_sqrs.append(detla_sqr)
            sum_delta_sqrs = sum(detla_sqrs)
            unc_pdf = math.sqrt(sum_delta_sqrs)
            unc_pdfs.append(unc_pdf)

        hist=HistStack(nested_dic_predict_sort['raw'], score_key, 'weight', nbin, 0, 1, "DNN score") 
        c1=hist.draw_hist(model_mass, ERA, file_name=out_dir+"/sys/normpdf"+model_mass)
        # print(hist.get_yields_by_bin()[model_mass])

        pdf_norm = np.array(hist.get_yields_by_bin()[key])
        # weight
        unc_pdfs1 = np.array(unc_pdfs)*pdf_norm[-1]/dic_yields["pdf0"][-1]

#         print('process:', p, "mass:", model_mass, "norm:", pdf_norm[-1], dic_yields["pdf0"][-1], "new:", unc_pdfs1[-1], unc_pdfs[-1])
        print('process:', p, "mass:", model_mass, "norm:", pdf_norm[-1], "pdf unc. scaled to norm:", unc_pdfs1[-1])

        pdf_up = pdf_norm + unc_pdfs1
        pdf_down = pdf_norm - unc_pdfs1

        h_pdfup = rt.TH1D(str_h+"_pdfUp", str_h+"_pdfUp", nbin, 0, 1)
        h_pdfdown = rt.TH1D(str_h+"_pdfDown", str_h+"_pdfDown", nbin, 0, 1)
        for nb in range(nbin):
            h_pdfup.SetBinContent(nb+1, pdf_up[nb])
            h_pdfdown.SetBinContent(nb+1, pdf_down[nb])
            
        write_sys_hist(h_pdfup, h_pdfdown)
        
        ##-- muR, muF unc. --##
        hists = []       
        for j in range(6):
            h = rt.TH1D("", "", nbin, 0, 1)
            for i, v in enumerate(nested_dic_predict_sort['raw'][key][score_key]):
                w = w0[i]*nested_dic_predict_sort['raw'][key]['scale'+str(j+1)+'_pre'][i]
                h.Fill(v, w)
        #     h.Draw("same")
            hists.append(h)        
        dic_yields=defaultdict(list)
        for i, h in enumerate(hists):
            for nb in range(nbin):
                dic_yields["scale"+str(i+1)].append(h.GetBinContent(nb+1))
                
        scale_up = np.maximum.reduce([np.array(dic_yields['scale1']), np.array(dic_yields['scale2']), np.array(dic_yields['scale3']), np.array(dic_yields['scale4']), np.array(dic_yields['scale5']), np.array(dic_yields['scale6'])])     
        scale_down = np.minimum.reduce([np.array(dic_yields['scale1']), np.array(dic_yields['scale2']), np.array(dic_yields['scale3']), np.array(dic_yields['scale4']), np.array(dic_yields['scale5']), np.array(dic_yields['scale6'])])  
        
        if p=='zp':
            scale_up = np.array(dic_yields['scale4'])
            scale_down = np.array(dic_yields['scale5'])
                                  
        print("envelope max.: ", scale_up)        
        print("envelope min.: ", scale_down)

        h_muRFup = rt.TH1D(str_h+"_muRFUp", str_h+"_muRFUp", nbin, 0, 1)
        h_muRFdown = rt.TH1D(str_h+"_muRFDown", str_h+"_muRFDown", nbin, 0, 1)
        for nb in range(nbin):
            h_muRFup.SetBinContent(nb+1, scale_up[nb])
            h_muRFdown.SetBinContent(nb+1, scale_down[nb])
            
        write_sys_hist(h_muRFup, h_muRFdown)
        
        ##-- ISR, FSR --##
        hists = []       
        for j in range(4):
            h = rt.TH1D("", "", nbin, 0, 1)
            for i, v in enumerate(nested_dic_predict_sort['raw'][key][score_key]):
                w = w0[i]*nested_dic_predict_sort['raw'][key]['psweight'+str(j+1)+'_pre'][i]
                h.Fill(v, w)
        #     h.Draw("same")
            hists.append(h)        
        dic_yields=defaultdict(list)
        for i, h in enumerate(hists):
            for nb in range(nbin):
                dic_yields["psweight"+str(i+1)].append(h.GetBinContent(nb+1))        

        h_isrup = rt.TH1D(str_h+"_isrUp", str_h+"_isrUp", nbin, 0, 1)
        h_isrdown = rt.TH1D(str_h+"_isrDown", str_h+"_isrDown", nbin, 0, 1)
        for nb in range(nbin):
            h_isrup.SetBinContent(nb+1, dic_yields["psweight1"][nb])
            h_isrdown.SetBinContent(nb+1, dic_yields["psweight2"][nb])
        write_sys_hist(h_isrup, h_isrdown)
        
        h_fsrup = rt.TH1D(str_h+"_fsrUp", str_h+"_fsrUp", nbin, 0, 1)
        h_fsrdown = rt.TH1D(str_h+"_fsrDown", str_h+"_fsrDown", nbin, 0, 1)
        for nb in range(nbin):
            h_fsrup.SetBinContent(nb+1, dic_yields["psweight3"][nb])
            h_fsrdown.SetBinContent(nb+1, dic_yields["psweight4"][nb])            
        write_sys_hist(h_fsrup, h_fsrdown)    

process: TT mass: Zp-2500_CH-345 norm: 7.710108550265431 pdf unc. scaled to norm: 0.16541545624180606
